# Experiment: SpectralBio Block 5 - Clinical Panel Audit (H100)

Objective:
- Translate the Block 2 failure regime into a reviewer-facing clinical panel audit.
- Reuse frozen multigene evidence, same-surface ESM-1v augmentations, and the Block 3 structure bridge when available.
- Surface a bounded but memorable class of panel variants where covariance rescues signal that scalar baselines under-call.
- Save a Lightning-friendly bundle under `New Notebooks/results/06_block5_clinical_panel_audit_h100/`.


## Block 5 Deliverables

- A bounded clinical-panel table with strong positive genes and explicit counterexamples.
- Same-surface comparisons against stronger scalar baselines when available, especially ESM-1v.
- A shortlist of rescue candidates for later Block 7 curation, with regime and structure hooks when available.
- Reviewer-friendly figures and `block6_summary.json` / `block6_summary.md` outputs.

## Runtime contract

- Target environment: **Lightning AI Studio**
- Target hardware: **H100**
- The notebook is designed to degrade honestly if upstream `01` / `02` / `03` / `05` outputs are missing.
- All outputs are written under `New Notebooks/results/06_block5_clinical_panel_audit_h100/`.


In [ ]:
# Setup: imports, identifiers, and notebook knobs
from __future__ import annotations

import importlib
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_SLUG = '06_block5_clinical_panel_audit_h100'
ACCOUNT_LABEL = os.environ.get('SPECTRALBIO_ACCOUNT_LABEL', 'SET_ACCOUNT_LABEL_HERE')
RUN_AT = datetime.now(timezone.utc).isoformat()
SEED = int(os.environ.get('SPECTRALBIO_RANDOM_SEED', '42'))
TOP_CASES = int(os.environ.get('SPECTRALBIO_BLOCK6_TOP_CASES', '12'))
PER_GENE_CASE_CAP = int(os.environ.get('SPECTRALBIO_BLOCK6_PER_GENE_CAP', '3'))
OVERWRITE = os.environ.get('SPECTRALBIO_OVERWRITE', '').strip().lower() in {'1', 'true', 'yes'}

def done(message: str) -> None:
    print(f'TERMINEI PODE SEGUIR - {message}')

print({
    'notebook_slug': NOTEBOOK_SLUG,
    'account_label': ACCOUNT_LABEL,
    'seed': SEED,
    'top_cases': TOP_CASES,
    'per_gene_case_cap': PER_GENE_CASE_CAP,
    'overwrite': OVERWRITE,
    'python': sys.version.split()[0],
    'platform': platform.platform(),
})
done('Initial configuration loaded.')


In [ ]:
# Helpers: subprocess, repo discovery, fallback loaders, and numeric utilities
def run(command: list[str], cwd: Path | None = None, check: bool = True) -> str:
    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=check,
        text=True,
        encoding='utf-8',
        errors='replace',
        capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed.stdout

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'notebooks').exists()

def find_repo_root() -> Path:
    env_repo = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()
    candidates: list[Path] = []
    if env_repo:
        candidates.append(Path(env_repo).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        Path('/teamspace/studios/this_studio/Stanford-Claw4s'),
        Path('/teamspace/studios/this_studio/SpectralBio'),
        Path('/content/Stanford-Claw4s'),
    ])
    for candidate in candidates:
        if looks_like_repo(candidate):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the SpectralBio repository root. Set SPECTRALBIO_REPO_ROOT or run inside the repo.')

def resolve_existing_path(raw: str | Path, repo_root: Path) -> Path:
    raw_path = Path(raw)
    if raw_path.exists():
        return raw_path.resolve()
    raw_text = str(raw).replace('\\', '/')
    for prefix in ('/content/Stanford-Claw4s/', '/teamspace/studios/this_studio/Stanford-Claw4s/', '/teamspace/studios/this_studio/SpectralBio/'):
        if raw_text.startswith(prefix):
            candidate = repo_root / raw_text[len(prefix):]
            if candidate.exists():
                return candidate.resolve()
    if 'Stanford-Claw4s/' in raw_text:
        candidate = repo_root / raw_text.split('Stanford-Claw4s/', 1)[1]
        if candidate.exists():
            return candidate.resolve()
    if not raw_path.is_absolute():
        candidate = (repo_root / raw_path).resolve()
        if candidate.exists():
            return candidate
    return raw_path

def maybe_read_csv(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

def binary_auc(labels: pd.Series, scores: pd.Series) -> float:
    frame = pd.DataFrame({
        'label': pd.to_numeric(labels, errors='coerce'),
        'score': pd.to_numeric(scores, errors='coerce'),
    }).dropna()
    frame = frame[frame['label'].isin([0, 1])].copy()
    if frame.empty:
        return float('nan')
    n_pos = int((frame['label'] == 1).sum())
    n_neg = int((frame['label'] == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return float('nan')
    ranks = frame['score'].rank(method='average')
    sum_pos = float(ranks[frame['label'] == 1].sum())
    return float((sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))

def percentile_rank(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors='coerce')
    return numeric.rank(method='average', pct=True)

def as_bool_series(series: pd.Series) -> pd.Series:
    return series.fillna('').astype(str).str.strip().str.lower().isin({'1', 'true', 'yes'})

def safe_float(value, default: float = float('nan')) -> float:
    try:
        return float(value)
    except Exception:
        return default

def variant_id_from_frame(frame: pd.DataFrame) -> pd.Series:
    positions = pd.to_numeric(frame['position'], errors='coerce').fillna(-1).astype(int).astype(str)
    return frame['gene'].astype(str) + ':' + frame['wt_aa'].astype(str) + positions + frame['mut_aa'].astype(str)

def median_or_nan(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors='coerce').dropna()
    if values.empty:
        return float('nan')
    return float(values.median())

def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None

def build_block1_summary_from_augmentations(repo_root: Path) -> pd.DataFrame:
    gene_paths = {
        'BRCA1': repo_root / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'brca1' / 'augmentation_table.csv',
        'BRCA2': repo_root / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'brca2' / 'augmentation_table.csv',
        'NSD1': repo_root / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'nsd1' / 'augmentation_table.csv',
        'TP53': repo_root / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'tp53' / 'augmentation_table.csv',
        'MSH2': repo_root / 'supplementary' / 'results-msh2' / 'msh2_esm1v_review_recovery_h100_results' / 'msh2_esm1v_review_recovery' / 'esm1v_augmentation' / 'msh2' / 'augmentation_table.csv',
    }
    rows = []
    for gene, path in gene_paths.items():
        if not path.exists():
            continue
        df = pd.read_csv(path)
        if df.empty:
            continue
        auc_esm1v = binary_auc(df['label'], df['esm1v_ll_mean_norm'])
        auc_reference_pair = binary_auc(df['label'], df['reference_pair_norm'])
        auc_augmented_pair = binary_auc(df['label'], df['augmented_pair_fixed_055'])
        rows.append({
            'gene': gene,
            'auc_esm1v_mean': auc_esm1v,
            'auc_reference_pair_fixed_055': auc_reference_pair,
            'auc_augmented_pair_fixed_055': auc_augmented_pair,
            'delta_augmented_vs_esm1v': auc_augmented_pair - auc_esm1v if not math.isnan(auc_augmented_pair) and not math.isnan(auc_esm1v) else float('nan'),
            'best_alpha_on_full_surface': float('nan'),
            'best_alpha_auc': float('nan'),
            'fixed_alpha_in_plateau': False,
            'plateau_alpha_min': float('nan'),
            'plateau_alpha_max': float('nan'),
            'paired_delta_ci_2p5': float('nan'),
            'paired_delta_ci_97p5': float('nan'),
            'summary_source': 'fallback_from_esm1v_augmentation_tables',
        })
    return pd.DataFrame(rows)

def build_block4_anchor_from_frozen(multigene_summary: pd.DataFrame, block1_summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in multigene_summary.to_dict(orient='records'):
        rows.append({
            'gene': row['gene'],
            'model_name': 'facebook/esm2_t30_150M_UR50D',
            'model_label': 'ESM2-150M (frozen)',
            'family_label': 'ESM2',
            'baseline_label': 'll_proper',
            'auc_baseline': safe_float(row.get('auc_ll_proper')),
            'auc_covariance': safe_float(row.get('auc_frob_dist')),
            'auc_pair_fixed_055': safe_float(row.get('auc_pair_fixed_055')),
            'delta_pair_vs_baseline': safe_float(row.get('delta_pair_vs_ll')),
            'source': 'fallback_multigene_summary',
            'n_total': safe_float(row.get('n_total')),
            'n_positive': safe_float(row.get('n_positive')),
            'n_negative': safe_float(row.get('n_negative')),
        })
    if not block1_summary.empty:
        for row in block1_summary.to_dict(orient='records'):
            rows.append({
                'gene': row['gene'],
                'model_name': 'facebook/esm1v_ensemble',
                'model_label': 'ESM-1v ensemble (frozen)',
                'family_label': 'ESM1v',
                'baseline_label': 'esm1v_mean',
                'auc_baseline': safe_float(row.get('auc_esm1v_mean')),
                'auc_covariance': float('nan'),
                'auc_pair_fixed_055': safe_float(row.get('auc_augmented_pair_fixed_055')),
                'delta_pair_vs_baseline': safe_float(row.get('delta_augmented_vs_esm1v')),
                'source': row.get('summary_source', 'fallback_block1_summary'),
                'n_total': float('nan'),
                'n_positive': float('nan'),
                'n_negative': float('nan'),
            })
    return pd.DataFrame(rows)

done('Helpers ready.')


In [ ]:
# Resolve repository root, outputs, and runtime dependencies
REPO_ROOT = find_repo_root()
RESULTS_ROOT = ensure_dir(REPO_ROOT / 'New Notebooks' / 'results' / NOTEBOOK_SLUG)
TABLES_DIR = ensure_dir(RESULTS_ROOT / 'tables')
FIGURES_DIR = ensure_dir(RESULTS_ROOT / 'figures')
TEXT_DIR = ensure_dir(RESULTS_ROOT / 'text')
MANIFESTS_DIR = ensure_dir(RESULTS_ROOT / 'manifests')
RUNTIME_DIR = ensure_dir(RESULTS_ROOT / 'runtime')
ZIP_PATH = REPO_ROOT / 'New Notebooks' / 'results' / f'{NOTEBOOK_SLUG}.zip'
ROOT_ZIP_COPY = REPO_ROOT / 'New Notebooks' / f'{NOTEBOOK_SLUG}.zip'

RUNTIME_REQUIREMENTS = [
    ('numpy', 'numpy==2.1.3'),
    ('pandas', 'pandas==2.2.3'),
    ('matplotlib', 'matplotlib==3.9.2'),
]
missing = []
for module_name, package_name in RUNTIME_REQUIREMENTS:
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(package_name)
if missing:
    run([sys.executable, '-m', 'pip', 'install', *missing], cwd=REPO_ROOT)
    importlib.invalidate_caches()

repo_status = {
    'repo_root': str(REPO_ROOT),
    'head_commit': run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT).strip(),
    'head_subject': run(['git', 'log', '-1', '--pretty=%s'], cwd=REPO_ROOT).strip(),
    'branch': run(['git', 'branch', '--show-current'], cwd=REPO_ROOT).strip(),
}
display(pd.DataFrame([repo_status]))
done('Repository, output directories, and dependencies confirmed.')


In [ ]:
# Load frozen evidence with safe fallbacks for upstream notebooks
panel_manifest_path = resolve_existing_path('supplementary/final_accept_a100/panel25/support_ranked_panel_manifest.json', REPO_ROOT)
multigene_summary_path = resolve_existing_path('supplementary/final_accept_a100/panel25/multigene/multigene_summary.csv', REPO_ROOT)
block1_summary_path = resolve_existing_path('New Notebooks/results/01_block1_baseline_alpha_regime_audit_h100/tables/reviewer_facing_summary_table.csv', REPO_ROOT)
block2_annotated_path = resolve_existing_path('New Notebooks/results/02_block2_failure_mode_hunt_h100/tables/annotated_failure_mode_rows.csv', REPO_ROOT)
block3_structure_path = resolve_existing_path('New Notebooks/results/05_block3_structure_bridge_h100/tables/structure_bridge_metrics.csv', REPO_ROOT)
block4_anchor_path = resolve_existing_path('New Notebooks/results/03_block4_model_agnostic_plms_h100_v2/tables/model_agnostic_anchor_long.csv', REPO_ROOT)

panel_manifest = json.loads(panel_manifest_path.read_text(encoding='utf-8'))
multigene_summary = pd.read_csv(multigene_summary_path)
for column in ['n_total', 'n_positive', 'n_negative', 'auc_ll_proper', 'auc_frob_dist', 'auc_pair_fixed_055', 'delta_pair_vs_ll']:
    if column in multigene_summary.columns:
        multigene_summary[column] = pd.to_numeric(multigene_summary[column], errors='coerce')

panel_gene_rows = []
for gene, payload in panel_manifest.get('genes', {}).items():
    panel_gene_rows.append({
        'gene': gene,
        'support_rank': payload.get('support_rank'),
        'panel_n_total': payload.get('n_total'),
        'panel_n_positive': payload.get('n_positive'),
        'panel_n_negative': payload.get('n_negative'),
        'sequence_source': payload.get('sequence_source'),
    })
panel_gene_manifest = pd.DataFrame(panel_gene_rows)

block1_summary = maybe_read_csv(block1_summary_path)
if block1_summary.empty:
    block1_summary = build_block1_summary_from_augmentations(REPO_ROOT)
else:
    block1_summary['summary_source'] = 'new_notebooks_block1_output'
for column in ['auc_esm1v_mean', 'auc_reference_pair_fixed_055', 'auc_augmented_pair_fixed_055', 'delta_augmented_vs_esm1v']:
    if column in block1_summary.columns:
        block1_summary[column] = pd.to_numeric(block1_summary[column], errors='coerce')
if 'summary_source' not in block1_summary.columns:
    block1_summary['summary_source'] = 'fallback_from_esm1v_augmentation_tables'

block2_annotated = maybe_read_csv(block2_annotated_path)
if not block2_annotated.empty:
    block2_annotated['variant_id'] = block2_annotated.get('variant_id', variant_id_from_frame(block2_annotated))
    if 'selected_regime' in block2_annotated.columns:
        block2_annotated['selected_regime'] = as_bool_series(block2_annotated['selected_regime'])
    if 'is_repair_exemplar' in block2_annotated.columns:
        block2_annotated['is_repair_exemplar'] = as_bool_series(block2_annotated['is_repair_exemplar'])
else:
    block2_annotated = pd.DataFrame(columns=['variant_id', 'selected_regime', 'is_repair_exemplar', 'context_signature', 'pair_repair', 'frob_repair', 'll_repair', 'validation_role'])

block3_structure = maybe_read_csv(block3_structure_path)
if not block3_structure.empty:
    block3_structure['variant_id'] = block3_structure.get('variant_id', variant_id_from_frame(block3_structure))
else:
    block3_structure = pd.DataFrame(columns=['variant_id', 'site_plddt', 'window_plddt_mean', 'site_contact_count', 'site_long_range_contact_count', 'cohort'])

block4_anchor = maybe_read_csv(block4_anchor_path)
if block4_anchor.empty:
    block4_anchor = build_block4_anchor_from_frozen(multigene_summary, block1_summary)
    block4_anchor['anchor_source'] = 'fallback_from_frozen_tables'
else:
    block4_anchor['anchor_source'] = 'new_notebooks_block4_output'
for column in ['auc_baseline', 'auc_covariance', 'auc_pair_fixed_055', 'delta_pair_vs_baseline', 'n_total', 'n_positive', 'n_negative']:
    if column in block4_anchor.columns:
        block4_anchor[column] = pd.to_numeric(block4_anchor[column], errors='coerce')

evidence_inventory = pd.DataFrame([
    {'artifact': 'panel_manifest', 'path': str(panel_manifest_path), 'exists': panel_manifest_path.exists(), 'rows': len(panel_gene_manifest), 'source': 'versioned'},
    {'artifact': 'multigene_summary', 'path': str(multigene_summary_path), 'exists': multigene_summary_path.exists(), 'rows': len(multigene_summary), 'source': 'versioned'},
    {'artifact': 'block1_summary', 'path': str(block1_summary_path), 'exists': block1_summary_path.exists(), 'rows': len(block1_summary), 'source': str(block1_summary['summary_source'].iloc[0]) if not block1_summary.empty else 'missing'},
    {'artifact': 'block2_annotated', 'path': str(block2_annotated_path), 'exists': block2_annotated_path.exists(), 'rows': len(block2_annotated), 'source': 'new_notebooks_or_empty'},
    {'artifact': 'block3_structure', 'path': str(block3_structure_path), 'exists': block3_structure_path.exists(), 'rows': len(block3_structure), 'source': 'new_notebooks_or_empty'},
    {'artifact': 'block4_anchor', 'path': str(block4_anchor_path), 'exists': block4_anchor_path.exists(), 'rows': len(block4_anchor), 'source': str(block4_anchor['anchor_source'].iloc[0]) if not block4_anchor.empty else 'missing'},
])
evidence_inventory.to_csv(TABLES_DIR / 'frozen_evidence_inventory.csv', index=False)
block1_summary.to_csv(TABLES_DIR / 'block1_summary_reused_or_fallback.csv', index=False)
block4_anchor.to_csv(TABLES_DIR / 'block4_anchor_reused_or_fallback.csv', index=False)
display(evidence_inventory)
done('Frozen evidence loaded with fallbacks recorded.')


In [ ]:
# Lock the bounded clinical panel with strong positives and explicit counterexamples
preferred_positive_order = ['BRCA2', 'TSC2', 'CREBBP', 'ANKRD11']
preferred_counterexample_order = ['BRCA1', 'MSH2']

gene_summary = multigene_summary.merge(panel_gene_manifest, on='gene', how='left')
gene_summary['support_rank'] = pd.to_numeric(gene_summary['support_rank'], errors='coerce')
gene_summary['delta_rank_desc'] = gene_summary['delta_pair_vs_ll'].rank(ascending=False, method='dense')

available_genes = set(gene_summary['gene'].tolist())
positive_genes = [gene for gene in preferred_positive_order if gene in available_genes]
counterexample_genes = [gene for gene in preferred_counterexample_order if gene in available_genes]

if len(positive_genes) < 4:
    fallback_candidates = gene_summary[
        (gene_summary['n_positive'] >= 30)
        & (gene_summary['n_negative'] >= 100)
        & (~gene_summary['gene'].isin(set(positive_genes + counterexample_genes)))
    ].sort_values(['delta_pair_vs_ll', 'n_total'], ascending=[False, False])
    for gene in fallback_candidates['gene'].tolist():
        positive_genes.append(gene)
        if len(positive_genes) >= 4:
            break

selected_genes = positive_genes + counterexample_genes
clinical_panel = gene_summary[gene_summary['gene'].isin(selected_genes)].copy()
clinical_panel['focus_role'] = np.where(
    clinical_panel['gene'].isin(positive_genes),
    'positive_focus',
    'counterexample',
)
clinical_panel['selection_reason'] = clinical_panel['gene'].map({
    'BRCA2': 'clinical_positive_focus',
    'TSC2': 'clinical_positive_focus',
    'CREBBP': 'clinical_positive_focus',
    'ANKRD11': 'clinical_positive_focus',
    'BRCA1': 'clinical_counterexample',
    'MSH2': 'clinical_counterexample',
}).fillna('fallback_positive_focus')

clinical_panel = clinical_panel.merge(
    block1_summary[['gene', 'auc_esm1v_mean', 'auc_reference_pair_fixed_055', 'auc_augmented_pair_fixed_055', 'delta_augmented_vs_esm1v', 'summary_source']].drop_duplicates(subset=['gene']),
    on='gene',
    how='left',
)

cross_model_support = (
    block4_anchor
    .groupby('gene', dropna=False)
    .agg(
        best_available_cross_model_delta=('delta_pair_vs_baseline', 'max'),
        n_positive_models=('delta_pair_vs_baseline', lambda s: int((pd.to_numeric(s, errors='coerce') > 0).sum())),
    )
    .reset_index()
)
non_esm2_positive = pd.DataFrame(columns=['gene', 'n_non_esm2_positive_models'])
if not block4_anchor.empty:
    non_esm2_positive = (
        block4_anchor.assign(
            family_label=block4_anchor['family_label'].fillna('unknown'),
            delta_pair_vs_baseline=pd.to_numeric(block4_anchor['delta_pair_vs_baseline'], errors='coerce'),
        )
        .query("family_label != 'ESM2' and delta_pair_vs_baseline > 0")
        .groupby('gene')
        .size()
        .rename('n_non_esm2_positive_models')
        .reset_index()
    )
clinical_panel = clinical_panel.merge(cross_model_support, on='gene', how='left')
clinical_panel = clinical_panel.merge(non_esm2_positive, on='gene', how='left')
clinical_panel['n_non_esm2_positive_models'] = clinical_panel['n_non_esm2_positive_models'].fillna(0).astype(int)
clinical_panel['same_surface_status'] = np.where(
    clinical_panel['delta_augmented_vs_esm1v'].isna(),
    'not_available',
    np.where(clinical_panel['delta_augmented_vs_esm1v'] > 0.03, 'covariance_wins', np.where(clinical_panel['delta_augmented_vs_esm1v'] < -0.03, 'scalar_wins', 'mixed')),
)
clinical_panel['esm2_status'] = np.where(
    clinical_panel['delta_pair_vs_ll'] > 0.05,
    'covariance_wins',
    np.where(clinical_panel['delta_pair_vs_ll'] < 0.0, 'scalar_wins', 'mixed'),
)
clinical_panel = clinical_panel.sort_values(['focus_role', 'delta_pair_vs_ll', 'n_total'], ascending=[True, False, False]).reset_index(drop=True)
clinical_panel.to_csv(TABLES_DIR / 'clinical_gene_panel.csv', index=False)
display(clinical_panel[['gene', 'focus_role', 'n_total', 'n_positive', 'n_negative', 'auc_ll_proper', 'auc_pair_fixed_055', 'delta_pair_vs_ll', 'auc_esm1v_mean', 'auc_augmented_pair_fixed_055', 'delta_augmented_vs_esm1v', 'same_surface_status', 'best_available_cross_model_delta']])
done('Clinical panel locked with positive genes and counterexamples.')


In [ ]:
# Load per-gene score tables, merge same-surface baselines, and rank rescue candidates
augmentation_paths = [
    REPO_ROOT / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'brca1' / 'augmentation_table.csv',
    REPO_ROOT / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'brca2' / 'augmentation_table.csv',
    REPO_ROOT / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'nsd1' / 'augmentation_table.csv',
    REPO_ROOT / 'supplementary' / 'final_accept_part3_esm1v_augmentation' / 'esm1v_augmentation' / 'tp53' / 'augmentation_table.csv',
    REPO_ROOT / 'supplementary' / 'results-msh2' / 'msh2_esm1v_review_recovery_h100_results' / 'msh2_esm1v_review_recovery' / 'esm1v_augmentation' / 'msh2' / 'augmentation_table.csv',
]
augmentation_tables = []
for path in augmentation_paths:
    if not path.exists():
        continue
    df = pd.read_csv(path)
    if df.empty:
        continue
    df['variant_id'] = variant_id_from_frame(df)
    for column in ['esm1v_ll_mean_norm', 'reference_pair_norm', 'augmented_pair_fixed_055', 'augmented_pair_facebook_esm2_t36_3B_UR50D']:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')
    augmentation_tables.append(df)
augmentation_long = pd.concat(augmentation_tables, ignore_index=True, sort=False) if augmentation_tables else pd.DataFrame(columns=['variant_id'])
augmentation_long.to_csv(TABLES_DIR / 'augmentation_long_reused_or_fallback.csv', index=False)

block2_subset = block2_annotated[['variant_id', 'selected_regime', 'is_repair_exemplar', 'context_signature', 'pair_repair', 'frob_repair', 'll_repair', 'validation_role']].drop_duplicates(subset=['variant_id']) if not block2_annotated.empty else pd.DataFrame(columns=['variant_id'])
block3_subset = block3_structure[['variant_id', 'site_plddt', 'window_plddt_mean', 'site_contact_count', 'site_long_range_contact_count', 'cohort']].drop_duplicates(subset=['variant_id']) if not block3_structure.empty else pd.DataFrame(columns=['variant_id'])

score_frames = []
score_inventory_rows = []
for gene in clinical_panel['gene'].tolist():
    gene_dir = REPO_ROOT / 'supplementary' / 'final_accept_a100' / 'panel25' / 'multigene' / gene.lower()
    score_path = first_existing(sorted(gene_dir.glob('*_facebook_esm2_t30_150M_UR50D_scores.csv')) if gene_dir.exists() else [])
    score_inventory_rows.append({'gene': gene, 'score_path': str(score_path) if score_path else '', 'exists': score_path is not None})
    if score_path is None:
        continue
    df = pd.read_csv(score_path)
    if df.empty:
        continue
    for column in ['position', 'label', 'frob_dist', 'trace_ratio', 'sps_log', 'll_proper']:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')
    df['variant_id'] = variant_id_from_frame(df)
    df['ll_rank_norm'] = percentile_rank(df['ll_proper'])
    df['frob_rank_norm'] = percentile_rank(df['frob_dist'])
    df['pair_rank_fixed_055'] = 0.55 * df['frob_rank_norm'] + 0.45 * df['ll_rank_norm']
    if not augmentation_long.empty:
        df = df.merge(
            augmentation_long[['variant_id', 'esm1v_ll_mean_norm', 'reference_pair_norm', 'augmented_pair_fixed_055', 'augmented_pair_facebook_esm2_t36_3B_UR50D']].drop_duplicates(subset=['variant_id']),
            on='variant_id',
            how='left',
        )
    df = df.merge(block2_subset, on='variant_id', how='left')
    df = df.merge(block3_subset, on='variant_id', how='left')
    if 'selected_regime' in df.columns:
        df['selected_regime'] = as_bool_series(df['selected_regime'])
    if 'is_repair_exemplar' in df.columns:
        df['is_repair_exemplar'] = as_bool_series(df['is_repair_exemplar'])
    df['scalar_baseline_for_audit'] = np.where(df.get('esm1v_ll_mean_norm', pd.Series(index=df.index, dtype=float)).notna(), df['esm1v_ll_mean_norm'], df['ll_rank_norm'])
    df['pair_for_audit'] = np.where(df.get('augmented_pair_fixed_055', pd.Series(index=df.index, dtype=float)).notna(), df['augmented_pair_fixed_055'], df['pair_rank_fixed_055'])
    df['covariance_only_gain'] = df['pair_rank_fixed_055'] - df['ll_rank_norm']
    df['clinical_rescue_margin'] = df['pair_for_audit'] - df['scalar_baseline_for_audit']
    site_contact_series = pd.to_numeric(df['site_contact_count'], errors='coerce') if 'site_contact_count' in df.columns else pd.Series(np.nan, index=df.index)
    window_plddt_series = pd.to_numeric(df['window_plddt_mean'], errors='coerce') if 'window_plddt_mean' in df.columns else pd.Series(np.nan, index=df.index)
    df['structure_support'] = np.where(
        site_contact_series.fillna(0) >= 7,
        'contact_dense',
        np.where(window_plddt_series.fillna(0) >= 80, 'confident_structure', 'not_available_or_weak'),
    )
    df['gene_case_rank'] = df['clinical_rescue_margin'].rank(ascending=False, method='dense')
    score_frames.append(df)

score_inventory = pd.DataFrame(score_inventory_rows)
combined_scores = pd.concat(score_frames, ignore_index=True, sort=False) if score_frames else pd.DataFrame()
score_inventory.to_csv(TABLES_DIR / 'clinical_score_inventory.csv', index=False)
combined_scores.to_csv(TABLES_DIR / 'clinical_panel_variant_scores.csv', index=False)
display(score_inventory)
display(combined_scores[['gene', 'variant_id', 'label', 'll_rank_norm', 'pair_rank_fixed_055', 'esm1v_ll_mean_norm', 'augmented_pair_fixed_055', 'clinical_rescue_margin', 'selected_regime', 'is_repair_exemplar']].head(10) if not combined_scores.empty else pd.DataFrame())
done('Per-gene score tables merged with same-surface and regime evidence.')


In [ ]:
# Build reviewer-facing clinical tables and the rescue shortlist
if combined_scores.empty:
    raise RuntimeError('No clinical score tables could be loaded. Stopping before any empty-claim output.')

positive_focus_set = set(positive_genes)
counterexample_set = set(counterexample_genes)

clinical_gene_table = clinical_panel[['gene', 'focus_role', 'selection_reason', 'support_rank', 'n_total', 'n_positive', 'n_negative', 'auc_ll_proper', 'auc_pair_fixed_055', 'delta_pair_vs_ll', 'auc_esm1v_mean', 'auc_augmented_pair_fixed_055', 'delta_augmented_vs_esm1v', 'same_surface_status', 'best_available_cross_model_delta', 'n_positive_models', 'n_non_esm2_positive_models']].copy()
clinical_gene_table = clinical_gene_table.sort_values(['focus_role', 'delta_pair_vs_ll', 'gene'], ascending=[True, False, True]).reset_index(drop=True)
clinical_gene_table.to_csv(TABLES_DIR / 'clinical_gene_audit_table.csv', index=False)

candidate_shortlist = combined_scores[
    (combined_scores['gene'].isin(positive_focus_set))
    & (pd.to_numeric(combined_scores['label'], errors='coerce') == 1)
    & (pd.to_numeric(combined_scores['pair_for_audit'], errors='coerce') >= 0.65)
    & (pd.to_numeric(combined_scores['clinical_rescue_margin'], errors='coerce') >= 0.12)
].copy()
if len(candidate_shortlist) < 5:
    candidate_shortlist = combined_scores[
        (combined_scores['gene'].isin(positive_focus_set))
        & (pd.to_numeric(combined_scores['label'], errors='coerce') == 1)
        & (pd.to_numeric(combined_scores['pair_for_audit'], errors='coerce') >= 0.60)
        & (pd.to_numeric(combined_scores['clinical_rescue_margin'], errors='coerce') >= 0.08)
    ].copy()
candidate_shortlist = candidate_shortlist.sort_values(
    ['clinical_rescue_margin', 'selected_regime', 'is_repair_exemplar', 'pair_for_audit'],
    ascending=[False, False, False, False],
)
candidate_shortlist = candidate_shortlist.groupby('gene', group_keys=False).head(PER_GENE_CASE_CAP).head(TOP_CASES).reset_index(drop=True)
candidate_shortlist['shortlist_reason'] = np.where(
    candidate_shortlist['selected_regime'].fillna(False),
    'block2_failure_regime_overlap',
    'clinical_margin_only',
)
candidate_shortlist.to_csv(TABLES_DIR / 'clinical_case_shortlist.csv', index=False)
candidate_shortlist.to_csv(TABLES_DIR / 'clinical_case_shortlist_for_block7.csv', index=False)

counterexample_cases = combined_scores[
    (combined_scores['gene'].isin(counterexample_set))
    & (pd.to_numeric(combined_scores['label'], errors='coerce') == 1)
].copy()
counterexample_cases = counterexample_cases.sort_values(['clinical_rescue_margin', 'pair_for_audit'], ascending=[True, False]).groupby('gene', group_keys=False).head(5).reset_index(drop=True)
counterexample_cases.to_csv(TABLES_DIR / 'clinical_counterexample_cases.csv', index=False)

shortlist_context_summary = (
    candidate_shortlist.assign(context_signature=candidate_shortlist['context_signature'].fillna('unannotated'))
    .groupby(['gene', 'context_signature'], dropna=False)
    .size()
    .rename('n_cases')
    .reset_index()
    .sort_values(['n_cases', 'gene', 'context_signature'], ascending=[False, True, True])
)
shortlist_context_summary.to_csv(TABLES_DIR / 'clinical_shortlist_context_summary.csv', index=False)

gene_case_summary = (
    combined_scores.assign(label=pd.to_numeric(combined_scores['label'], errors='coerce'))
    .groupby('gene', dropna=False)
    .agg(
        n_rows=('variant_id', 'count'),
        n_positive=('label', lambda s: int((s == 1).sum())),
        median_clinical_rescue_margin=('clinical_rescue_margin', median_or_nan),
        median_pair_for_audit=('pair_for_audit', median_or_nan),
        median_scalar_baseline=('scalar_baseline_for_audit', median_or_nan),
        n_regime_hits=('selected_regime', lambda s: int(as_bool_series(s).sum())),
    )
    .reset_index()
)
if not candidate_shortlist.empty:
    shortlist_counts = candidate_shortlist.groupby('gene').size().rename('n_shortlisted').reset_index()
    gene_case_summary = gene_case_summary.merge(shortlist_counts, on='gene', how='left')
else:
    gene_case_summary['n_shortlisted'] = 0
gene_case_summary['n_shortlisted'] = gene_case_summary['n_shortlisted'].fillna(0).astype(int)
gene_case_summary.to_csv(TABLES_DIR / 'clinical_gene_case_summary.csv', index=False)

display(clinical_gene_table)
display(candidate_shortlist[['gene', 'variant_id', 'name', 'clinical_rescue_margin', 'pair_for_audit', 'scalar_baseline_for_audit', 'selected_regime', 'is_repair_exemplar', 'structure_support']])
display(counterexample_cases[['gene', 'variant_id', 'name', 'clinical_rescue_margin', 'pair_for_audit', 'scalar_baseline_for_audit']])
done('Clinical tables, shortlist, and counterexamples saved.')


In [ ]:
# Draw reviewer-facing figures for the bounded clinical-panel claim
focus_color_map = {'positive_focus': '#1f77b4', 'counterexample': '#d62728'}

fig, ax = plt.subplots(figsize=(9, 5))
ordered = clinical_gene_table.sort_values(['focus_role', 'delta_pair_vs_ll', 'gene'], ascending=[True, False, True]).copy()
bars = ax.bar(ordered['gene'], ordered['delta_pair_vs_ll'], color=[focus_color_map.get(role, '#7f7f7f') for role in ordered['focus_role']])
ax.axhline(0.0, color='black', linewidth=1.0)
ax.set_title('Clinical panel: covariance pair vs scalar ESM2 baseline')
ax.set_ylabel('delta AUC (pair - scalar baseline)')
ax.set_xlabel('gene')
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.25)
for bar, value in zip(bars, ordered['delta_pair_vs_ll']):
    ax.text(bar.get_x() + bar.get_width() / 2, value + (0.005 if value >= 0 else -0.015), f'{value:.3f}', ha='center', va='bottom' if value >= 0 else 'top', fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'clinical_panel_delta_auc.png', dpi=220)
plt.close(fig)

same_surface = clinical_gene_table[clinical_gene_table['delta_augmented_vs_esm1v'].notna()].copy()
if not same_surface.empty:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    bars = ax.bar(same_surface['gene'], same_surface['delta_augmented_vs_esm1v'], color=['#2ca02c' if value > 0 else '#9467bd' for value in same_surface['delta_augmented_vs_esm1v']])
    ax.axhline(0.0, color='black', linewidth=1.0)
    ax.set_title('Same-surface audit: covariance pair vs ESM-1v scalar baseline')
    ax.set_ylabel('delta AUC (augmented pair - ESM-1v mean)')
    ax.set_xlabel('gene')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.25)
    for bar, value in zip(bars, same_surface['delta_augmented_vs_esm1v']):
        ax.text(bar.get_x() + bar.get_width() / 2, value + (0.004 if value >= 0 else -0.012), f'{value:.3f}', ha='center', va='bottom' if value >= 0 else 'top', fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'same_surface_delta_auc_vs_esm1v.png', dpi=220)
    plt.close(fig)

if not candidate_shortlist.empty:
    shortlist_plot = candidate_shortlist.copy().sort_values(['clinical_rescue_margin', 'gene'], ascending=[False, True]).head(min(len(candidate_shortlist), 12))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(shortlist_plot['variant_id'], shortlist_plot['clinical_rescue_margin'], color='#ff7f0e')
    ax.axvline(0.0, color='black', linewidth=1.0)
    ax.set_title('Top clinical rescue candidates')
    ax.set_xlabel('pair score - scalar baseline score')
    ax.set_ylabel('variant')
    ax.grid(axis='x', alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'clinical_rescue_shortlist.png', dpi=220)
    plt.close(fig)

if not shortlist_context_summary.empty:
    context_plot = shortlist_context_summary.groupby('context_signature', dropna=False)['n_cases'].sum().sort_values(ascending=False).head(8)
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.bar(context_plot.index.astype(str), context_plot.values, color='#8c564b')
    ax.set_title('Shortlist contexts enriched among rescue candidates')
    ax.set_ylabel('n shortlisted cases')
    ax.set_xlabel('context signature')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'clinical_shortlist_contexts.png', dpi=220)
    plt.close(fig)

print('Saved figures:', sorted(path.name for path in FIGURES_DIR.glob('*.png')))
done('Reviewer-facing clinical figures saved.')


In [ ]:
# Write markdown/json summaries, a claim paragraph, and the final bundle
positive_focus_gene_wins = int((clinical_gene_table['focus_role'].eq('positive_focus') & (pd.to_numeric(clinical_gene_table['delta_pair_vs_ll'], errors='coerce') >= 0.05)).sum())
same_surface_wins = int((pd.to_numeric(clinical_gene_table['delta_augmented_vs_esm1v'], errors='coerce') >= 0.03).sum())
counterexample_losses = int((clinical_gene_table['focus_role'].eq('counterexample') & ((pd.to_numeric(clinical_gene_table['delta_pair_vs_ll'], errors='coerce') <= 0.01) | (pd.to_numeric(clinical_gene_table['delta_augmented_vs_esm1v'], errors='coerce') < 0))).sum())
shortlisted_cases = int(len(candidate_shortlist))
shortlist_gene_count = int(candidate_shortlist['gene'].nunique()) if not candidate_shortlist.empty else 0
best_shortlist_case = candidate_shortlist.iloc[0].to_dict() if not candidate_shortlist.empty else {}

claim_status = 'weak_clinical_panel_audit'
claim_reason = 'The clinical panel remains descriptive; keep the section as bounded supporting evidence only.'
if positive_focus_gene_wins >= 3 and shortlisted_cases >= 5 and shortlist_gene_count >= 3 and counterexample_losses >= 1:
    claim_status = 'clinical_panel_failure_regime_supported'
    claim_reason = 'Multiple clinical genes improve with covariance, the shortlist spans several genes, and BRCA1/MSH2-style counterexamples keep the claim bounded instead of over-generalized.'
elif positive_focus_gene_wins >= 2 and shortlisted_cases >= 3:
    claim_status = 'bounded_clinical_panel_audit_supported'
    claim_reason = 'The panel shows a bounded but reusable failure regime in clinical genes, with a shortlist ready for Block 7 curation even though the effect is not universal.'
elif shortlisted_cases >= 2:
    claim_status = 'case_driven_clinical_audit'
    claim_reason = 'Gene-level gains are mixed, but the notebook still surfaces clinical rescue cases that can anchor a case-study narrative.'

claim_paragraph = (
    'In a bounded clinical-panel audit, covariance-aware scoring improves over the frozen ESM2 scalar baseline in '
    f'{positive_focus_gene_wins} of {len(positive_genes)} positive-focus genes, while explicitly failing or flattening in '
    f'{counterexample_losses} counterexample settings. The audit yields {shortlisted_cases} rescue candidates across '
    f'{shortlist_gene_count} genes for downstream Block 7 curation. These shortlist entries are not asserted to be final VUS exemplars yet; '
    'they are prioritized ClinVar-style missense cases where covariance raises signal that scalar baselines under-call.'
)

summary_payload = {
    'notebook_slug': NOTEBOOK_SLUG,
    'account_label': ACCOUNT_LABEL,
    'run_at_utc': RUN_AT,
    'head_commit': repo_status['head_commit'],
    'claim_status': claim_status,
    'claim_reason': claim_reason,
    'positive_focus_genes': positive_genes,
    'counterexample_genes': counterexample_genes,
    'positive_focus_gene_wins': positive_focus_gene_wins,
    'same_surface_wins': same_surface_wins,
    'counterexample_losses': counterexample_losses,
    'shortlisted_cases': shortlisted_cases,
    'shortlist_gene_count': shortlist_gene_count,
    'best_shortlist_case': best_shortlist_case,
    'claim_paragraph': claim_paragraph,
}
(MANIFESTS_DIR / 'block6_summary.json').write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')
(MANIFESTS_DIR / 'artifact_summary.json').write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')

summary_md = '\n'.join([
    '# Block 5 Clinical Panel Audit Summary',
    '',
    f"- Claim status: `{claim_status}`",
    f"- Claim reason: {claim_reason}",
    f"- Positive-focus genes with delta >= 0.05: `{positive_focus_gene_wins}` of `{len(positive_genes)}`",
    f"- Same-surface wins against ESM-1v where available: `{same_surface_wins}`",
    f"- Counterexample losses or flat settings: `{counterexample_losses}`",
    f"- Shortlisted rescue candidates: `{shortlisted_cases}` across `{shortlist_gene_count}` genes",
    '',
    '## Claim Paragraph',
    '',
    claim_paragraph,
])
(TEXT_DIR / 'block6_summary.md').write_text(summary_md + '\n', encoding='utf-8')
(TEXT_DIR / 'clinical_claim_paragraph.md').write_text(claim_paragraph + '\n', encoding='utf-8')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
if ROOT_ZIP_COPY.exists():
    ROOT_ZIP_COPY.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_ROOT.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(RESULTS_ROOT.parent))
shutil.copy2(ZIP_PATH, ROOT_ZIP_COPY)
print(summary_md)
done('Block 5 clinical panel bundle is ready for download.')
